# 01 · Exploratory Data Analysis

Inspect the curated COCO subset: class distribution, box sizes, and sample images.

Run `uv run python main.py download-data --mode demo` first so `data/processed/yolo/` exists.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

from object_tracking_app.config.settings import get_settings
from object_tracking_app.utils.io import load_json

settings = get_settings()
print('Mode:', settings.dataset.mode)
print('Classes:', settings.dataset.classes)

## Load the split manifest

In [ ]:
splits_dir = settings.resolve(settings.dataset.splits_dir)
manifest_path = splits_dir / f'{settings.dataset.mode}_manifest.json'

if manifest_path.exists():
    manifest = load_json(manifest_path)
    print('Train images:', len(manifest['train']))
    print('Val images:', len(manifest['val']))
else:
    print('Manifest not found -- run scripts/download_dataset.py first:')
    print('  uv run python main.py download-data --mode', settings.dataset.mode)

## Class distribution across YOLO labels

In [ ]:
import glob
from collections import Counter

export_cfg = settings.dataset.yolo_export
output_root = settings.resolve(export_cfg.output_dir)
label_files = glob.glob(str(output_root / export_cfg.labels_train_subdir / '*.txt'))

class_counter = Counter()
for lf in label_files:
    with open(lf) as f:
        for line in f:
            if line.strip():
                cls_id = int(line.split()[0])
                class_counter[settings.dataset.classes[cls_id]] += 1

class_counter

In [ ]:
import matplotlib.pyplot as plt

if class_counter:
    names, counts = zip(*class_counter.most_common())
    plt.figure(figsize=(8, 4))
    plt.bar(names, counts)
    plt.xticks(rotation=45, ha='right')
    plt.title('Instances per class (train split)')
    plt.tight_layout()
    plt.show()
else:
    print('No labels found yet -- build the dataset subset first.')

## Preview a few annotated samples

In [ ]:
import cv2
import numpy as np

from object_tracking_app.data.transforms import xywhn_to_xyxy
from object_tracking_app.utils.viz import draw_detections

images_dir = output_root / export_cfg.train_subdir
labels_dir = output_root / export_cfg.labels_train_subdir

sample_images = sorted(images_dir.glob('*.jpg'))[:3]
for img_path in sample_images:
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    label_path = labels_dir / (img_path.stem + '.txt')
    boxes, names, scores = [], [], []
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            if not line.strip():
                continue
            cls_id, cx, cy, bw, bh = line.split()
            box = xywhn_to_xyxy(np.array([float(cx), float(cy), float(bw), float(bh)]), w, h)
            boxes.append(box)
            names.append(settings.dataset.classes[int(cls_id)])
            scores.append(1.0)
    if boxes:
        draw_detections(img, np.array(boxes), scores, names)
    plt.figure(figsize=(6, 4))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(img_path.name)
    plt.axis('off')
    plt.show()